# Daglab Troubleshooting Guide

This notebook helps diagnose and fix common issues when connecting to Dagster.

## 1. Check Environment and Dependencies

In [ ]:
import sys
import os
import importlib.util

print(f"Python version: {sys.version}")
print(f"Python path: {sys.executable}")
print(f"\nEnvironment variables:")

# Check Dagster-related environment variables
env_vars = [
    "DAGSTER_URL",
    "DAGSTER_TOKEN",
    "DAGSTER_USERNAME",
    "DAGSTER_PASSWORD",
    "DAGSTER_DEPLOYMENT",
    "DAGSTER_HOME"
]

for var in env_vars:
    value = os.getenv(var)
    if value:
        # Mask sensitive values
        if "TOKEN" in var or "PASSWORD" in var:
            print(f"  {var}: {'*' * 8}")
        else:
            print(f"  {var}: {value}")
    else:
        print(f"  {var}: Not set")

In [ ]:
# Check required packages
required_packages = {
    "daglab": "Daglab core",
    "dagster": "Dagster core",
    "dagster_graphql": "Dagster GraphQL client",
    "httpx": "HTTP client",
    "pandas": "Data manipulation",
    "marimo": "Notebook runtime"
}

print("Package status:")
for package, description in required_packages.items():
    spec = importlib.util.find_spec(package)
    if spec:
        print(f"  ✅ {package}: {description} - Installed")
        # Try to get version
        try:
            module = __import__(package)
            if hasattr(module, "__version__"):
                print(f"     Version: {module.__version__}")
        except:
            pass
    else:
        print(f"  ❌ {package}: {description} - Not installed")

## 2. Test Basic Connectivity

In [ ]:
# Test basic HTTP connectivity
import httpx
from urllib.parse import urljoin

dagster_url = os.getenv("DAGSTER_URL", "http://localhost:3000")
print(f"Testing connection to: {dagster_url}")

# Test base URL
try:
    response = httpx.get(dagster_url, timeout=10.0, follow_redirects=True)
    print(f"\n✅ Base URL reachable: {response.status_code}")
    print(f"   Content-Type: {response.headers.get('content-type', 'Unknown')}")
except httpx.TimeoutException:
    print(f"\n❌ Connection timeout - Dagster may not be running")
except httpx.NetworkError as e:
    print(f"\n❌ Network error: {e}")
except Exception as e:
    print(f"\n❌ Error: {e}")

# Test GraphQL endpoint
graphql_url = urljoin(dagster_url, "/graphql")
print(f"\nTesting GraphQL endpoint: {graphql_url}")

try:
    # Send a simple GraphQL query
    headers = {"Content-Type": "application/json"}
    
    # Add auth if available
    token = os.getenv("DAGSTER_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"
        print("  Using bearer token authentication")
    
    query = {"query": "{ __typename }"}
    response = httpx.post(graphql_url, json=query, headers=headers, timeout=10.0)
    
    print(f"\n✅ GraphQL endpoint reachable: {response.status_code}")
    
    if response.status_code == 200:
        data = response.json()
        print(f"   Response: {data}")
    else:
        print(f"   ❌ Unexpected status code")
        print(f"   Response: {response.text[:200]}")
        
except Exception as e:
    print(f"\n❌ GraphQL error: {e}")

## 3. Test Daglab GraphQL Client

In [ ]:
# Test Daglab GraphQL client
try:
    from daglab.helpers.graphql import DagsterClientSync, DagsterClientError
    from daglab.helpers.auth import AuthConfig, AuthType
    
    print("✅ Daglab GraphQL modules imported successfully")
    
    # Create auth config
    auth_config = AuthConfig.from_env()
    print(f"\nAuthentication type: {auth_config.auth_type.value}")
    
    # Create client
    client = DagsterClientSync(
        endpoint=graphql_url,
        auth_config=auth_config,
        timeout=30.0,
        verify_ssl=True
    )
    
    print("\nTesting health check...")
    is_healthy = client.health_check()
    
    if is_healthy:
        print("✅ Health check passed!")
    else:
        print("❌ Health check failed")
        
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\nTry installing Daglab:")
    print("  pip install daglab")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"\nError type: {type(e).__name__}")

## 4. Query Dagster Instance Information

In [ ]:
# Query for basic instance information
if 'client' in locals() and client:
    print("Querying Dagster instance...\n")
    
    # Version query
    try:
        version_query = """
        query {
            version
        }
        """
        result = client.query(version_query)
        print(f"Dagster version: {result.get('version', 'Unknown')}")
    except Exception as e:
        print(f"❌ Version query failed: {e}")
    
    # Instance info query
    try:
        instance_query = """
        query {
            instance {
                info
                runLauncher {
                    name
                }
                runQueuingSupported
            }
        }
        """
        result = client.query(instance_query)
        
        if 'instance' in result:
            instance = result['instance']
            print(f"\nInstance info: {instance.get('info', 'N/A')}")
            print(f"Run launcher: {instance.get('runLauncher', {}).get('name', 'N/A')}")
            print(f"Run queuing supported: {instance.get('runQueuingSupported', False)}")
    except Exception as e:
        print(f"❌ Instance query failed: {e}")
        
else:
    print("No client connection available")

## 5. Common Issues and Solutions

In [ ]:
# Diagnose common issues
print("Common Issues Checklist:\n")

issues = []

# Check 1: Dagster URL
if not os.getenv("DAGSTER_URL"):
    issues.append({
        "issue": "DAGSTER_URL not set",
        "solution": "Set DAGSTER_URL environment variable (e.g., export DAGSTER_URL=http://localhost:3000)"
    })

# Check 2: Authentication
if not (os.getenv("DAGSTER_TOKEN") or (os.getenv("DAGSTER_USERNAME") and os.getenv("DAGSTER_PASSWORD"))):
    issues.append({
        "issue": "No authentication configured",
        "solution": "Set DAGSTER_TOKEN or DAGSTER_USERNAME/DAGSTER_PASSWORD if your instance requires auth"
    })

# Check 3: Network connectivity
if 'localhost' in dagster_url or '127.0.0.1' in dagster_url:
    issues.append({
        "issue": "Using localhost URL",
        "solution": "Ensure Dagster is running locally, or update URL to remote instance"
    })

# Check 4: HTTPS vs HTTP
if dagster_url.startswith("https://") and 'localhost' in dagster_url:
    issues.append({
        "issue": "Using HTTPS with localhost",
        "solution": "Local Dagster typically uses HTTP. Try http://localhost:3000"
    })

if issues:
    for i, issue in enumerate(issues, 1):
        print(f"{i}. ⚠️ {issue['issue']}")
        print(f"   Solution: {issue['solution']}\n")
else:
    print("✅ No obvious configuration issues detected")

print("\n" + "="*50)
print("Additional Troubleshooting Steps:\n")
print("1. Verify Dagster is running:")
print("   - For local: `dagster dev` or `dagit`")
print("   - For Docker: `docker ps` to check containers")
print("\n2. Check Dagster logs for errors")
print("\n3. Try accessing Dagster UI in browser:")
print(f"   {dagster_url}")
print("\n4. For auth issues, verify token/credentials are correct")
print("\n5. For SSL issues, try setting verify_ssl=False (dev only)")

## 6. Test Repository Discovery

In [ ]:
# If we have a connection, try to discover repositories
if 'client' in locals() and client:
    print("Attempting to discover repositories...\n")
    
    discovery_query = """
    query {
        repositoriesOrError {
            __typename
            ... on RepositoryConnection {
                nodes {
                    name
                }
            }
            ... on PythonError {
                message
                stack
            }
        }
    }
    """
    
    try:
        result = client.query(discovery_query)
        
        if 'repositoriesOrError' in result:
            repos_or_error = result['repositoriesOrError']
            typename = repos_or_error.get('__typename')
            
            if typename == 'RepositoryConnection':
                repos = repos_or_error.get('nodes', [])
                if repos:
                    print(f"✅ Found {len(repos)} repositories:")
                    for repo in repos:
                        print(f"   - {repo['name']}")
                else:
                    print("⚠️ No repositories found")
                    print("\nPossible reasons:")
                    print("- No code locations configured")
                    print("- Repositories not loaded")
                    print("- Permission issues")
                    
            elif typename == 'PythonError':
                print(f"❌ Repository error: {repos_or_error.get('message')}")
                if 'stack' in repos_or_error:
                    print("\nStack trace:")
                    print(repos_or_error['stack'][:500] + "...")
            else:
                print(f"❓ Unexpected type: {typename}")
                
    except DagsterClientError as e:
        print(f"❌ GraphQL error: {e}")
        if hasattr(e, 'errors'):
            for error in e.errors:
                print(f"   - {error}")
                
    except Exception as e:
        print(f"❌ Unexpected error: {e}")
        print(f"   Type: {type(e).__name__}")

## 7. Security Validation Test

In [ ]:
# Test security validation
try:
    from daglab.validation.security import (
        validate_graphql_query,
        validate_run_config,
        validate_tags
    )
    
    print("Testing security validators...\n")
    
    # Test 1: Valid query
    valid_query = "query { repositoriesOrError { nodes { name } } }"
    is_valid, error = validate_graphql_query(valid_query)
    print(f"Valid query test: {'✅ PASS' if is_valid else '❌ FAIL'}")
    if error:
        print(f"  Error: {error}")
    
    # Test 2: Dangerous query
    dangerous_query = "query { __schema { types { name } } }"
    is_valid, error = validate_graphql_query(dangerous_query)
    print(f"\nDangerous query test: {'✅ PASS (blocked)' if not is_valid else '❌ FAIL (allowed)'}")
    if error:
        print(f"  Error: {error}")
    
    # Test 3: Run config validation
    test_config = {"ops": {}, "resources": {}}
    is_valid, error = validate_run_config(test_config)
    print(f"\nRun config test: {'✅ PASS' if is_valid else '❌ FAIL'}")
    if error:
        print(f"  Error: {error}")
    
    # Test 4: Tags validation
    test_tags = {"environment": "test", "user": "daglab"}
    is_valid, error = validate_tags(test_tags)
    print(f"\nTags test: {'✅ PASS' if is_valid else '❌ FAIL'}")
    if error:
        print(f"  Error: {error}")
        
except ImportError:
    print("❌ Security validation module not available")
    print("   This is expected if Daglab is not installed")

## Summary

In [ ]:
# Generate summary report
print("=" * 60)
print("TROUBLESHOOTING SUMMARY")
print("=" * 60)
print()

# Collect status
status_items = []

# Environment
if os.getenv("DAGSTER_URL"):
    status_items.append(("✅", "DAGSTER_URL configured"))
else:
    status_items.append(("❌", "DAGSTER_URL not set"))

# Authentication
if os.getenv("DAGSTER_TOKEN") or (os.getenv("DAGSTER_USERNAME") and os.getenv("DAGSTER_PASSWORD")):
    status_items.append(("✅", "Authentication configured"))
else:
    status_items.append(("⚠️", "No authentication configured"))

# Packages
try:
    import daglab
    status_items.append(("✅", "Daglab installed"))
except:
    status_items.append(("❌", "Daglab not installed"))

# Connection
if 'client' in locals() and client:
    if 'is_healthy' in locals() and is_healthy:
        status_items.append(("✅", "Connected to Dagster"))
    else:
        status_items.append(("❌", "Connection failed"))
else:
    status_items.append(("❌", "No client connection"))

# Print status
for icon, message in status_items:
    print(f"{icon} {message}")

print("\n" + "=" * 60)
print("\nNext steps:")
failed_items = [item for item in status_items if item[0] in ["❌", "⚠️"]]
if failed_items:
    print("1. Address the issues marked with ❌ above")
    print("2. Re-run this troubleshooting notebook")
    print("3. Check Dagster logs if issues persist")
else:
    print("✅ Everything looks good! You should be able to use Daglab notebooks.")
    print("\nTry running one of the example notebooks:")
    print("- minimal_example.py")
    print("- full_featured_example.py")